# v11 Settlement Two-Provider Experiment

This notebook trains and compares three v11 settlement-first variants:

- GFS + HRRR
- GFS + NBM
- NBM + HRRR

Each arm rebuilds features with only the selected providers, tunes its own XGBoost, LightGBM, and CatBoost base learners, tunes a ridge stack, and optionally exports production-refit bundles.

In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise RuntimeError('Could not locate the weather-research repository root.')


REPO_ROOT = find_repo_root(Path.cwd())
RUNNER = REPO_ROOT / 'scripts/run_station_stacking_v11_settlement_provider_pairs.py'
OUTPUT_ROOT = REPO_ROOT / 'data/calibration/station_stacking_v11_settlement_provider_pairs'
REPO_ROOT

WindowsPath('D:/dev/weather-research')

## Configuration

The defaults below run the full KATL/KDAL comparison. Set `FAST_MODE = True` and reduce trial counts for a pipeline check. Full optimization can take a long time.

In [2]:
STATIONS = ('KATL', 'KDAL')
VARIANTS = ('gfs_hrrr', 'gfs_nbm', 'nbm_hrrr')
OPTUNA_TRIALS = 30
STARTUP_TRIALS = 15
STACK_OPTUNA_TRIALS = 30
STACK_STARTUP_TRIALS = 15
FAST_MODE = False
EXPORT_MODELS = True

pd.DataFrame(
    [{'variant': name, 'providers': name.replace('_', ' + ').upper()} for name in VARIANTS]
)

,variant,providers
0,gfs_hrrr,GFS + HRRR
1,gfs_nbm,GFS + NBM
2,nbm_hrrr,NBM + HRRR


## Train all provider-pair arms

In [3]:
command = [
    sys.executable,
    str(RUNNER),
    '--stations', ','.join(STATIONS),
    '--variants', ','.join(VARIANTS),
    '--optuna-trials', str(OPTUNA_TRIALS),
    '--startup-trials', str(STARTUP_TRIALS),
    '--stack-optuna-trials', str(STACK_OPTUNA_TRIALS),
    '--stack-startup-trials', str(STACK_STARTUP_TRIALS),
    '--output-root', str(OUTPUT_ROOT),
    '--quiet-optuna',
]
if FAST_MODE:
    command.append('--fast-mode')
if not EXPORT_MODELS:
    command.append('--skip-export')

print(' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

d:\dev\weather-research\.venv\Scripts\python.exe D:\dev\weather-research\scripts\run_station_stacking_v11_settlement_provider_pairs.py --stations KATL,KDAL --variants gfs_hrrr,gfs_nbm,nbm_hrrr --optuna-trials 30 --startup-trials 15 --stack-optuna-trials 30 --stack-startup-trials 15 --output-root D:\dev\weather-research\data\calibration\station_stacking_v11_settlement_provider_pairs --quiet-optuna


CalledProcessError: Command '['d:\\dev\\weather-research\\.venv\\Scripts\\python.exe', 'D:\\dev\\weather-research\\scripts\\run_station_stacking_v11_settlement_provider_pairs.py', '--stations', 'KATL,KDAL', '--variants', 'gfs_hrrr,gfs_nbm,nbm_hrrr', '--optuna-trials', '30', '--startup-trials', '15', '--stack-optuna-trials', '30', '--stack-startup-trials', '15', '--output-root', 'D:\\dev\\weather-research\\data\\calibration\\station_stacking_v11_settlement_provider_pairs', '--quiet-optuna']' returned non-zero exit status 4294967295.

## Ridge-stack comparison

In [ ]:
comparison_path = OUTPUT_ROOT / 'provider_pair_comparison.csv'
comparison = pd.read_csv(comparison_path)
comparison = comparison.sort_values(['station_id', 'period', 'mae_f']).reset_index(drop=True)
display(comparison)

In [ ]:
test_comparison = comparison.loc[comparison['period'].eq('test_2026')].copy()
mae_table = test_comparison.pivot(index='station_id', columns='variant', values='mae_f')
rmse_table = test_comparison.pivot(index='station_id', columns='variant', values='rmse_f')

print('2026 ridge-stack MAE (°F)')
display(mae_table.style.format('{:.3f}').highlight_min(axis=1, color='#b7e4c7'))
print('2026 ridge-stack RMSE (°F)')
display(rmse_table.style.format('{:.3f}').highlight_min(axis=1, color='#b7e4c7'))

## Best pair by station

In [ ]:
best = (
    test_comparison.sort_values(['station_id', 'mae_f', 'rmse_f'])
    .groupby('station_id', as_index=False)
    .first()[['station_id', 'variant', 'providers', 'count', 'mae_f', 'rmse_f', 'model_version']]
)
display(best.style.format({'mae_f': '{:.3f}', 'rmse_f': '{:.3f}'}))

## Compare against the original three-provider v11 settlement model

In [ ]:
baseline_root = REPO_ROOT / 'data/calibration/station_stacking_v11_settlement'
baseline_rows = []
for station in STATIONS:
    path = baseline_root / f'{station}_year_split_scoreboard.csv'
    frame = pd.read_csv(path)
    selected = frame.loc[
        frame['period'].eq('test_2026') & frame['method'].eq('ridge_stack')
    ].copy()
    selected['station_id'] = station
    selected['variant'] = 'gfs_hrrr_nbm'
    selected['providers'] = 'gfs+hrrr+nbm'
    baseline_rows.append(selected)

baseline = pd.concat(baseline_rows, ignore_index=True)
combined = pd.concat(
    [
        test_comparison[['station_id', 'variant', 'providers', 'count', 'mae_f', 'rmse_f']],
        baseline[['station_id', 'variant', 'providers', 'count', 'mae_f', 'rmse_f']],
    ],
    ignore_index=True,
).sort_values(['station_id', 'mae_f'])
display(combined.style.format({'mae_f': '{:.3f}', 'rmse_f': '{:.3f}'}))

## Artifact checks

In [ ]:
artifact_rows = []
for variant in VARIANTS:
    for station in STATIONS:
        model_dir = OUTPUT_ROOT / variant / 'model_weights'
        bundles = list(model_dir.glob(f'{station}_*.joblib'))
        manifests = list(model_dir.glob(f'{station}_*.json'))
        artifact_rows.append(
            {
                'variant': variant,
                'station_id': station,
                'bundle_ok': len(bundles) == 1 if EXPORT_MODELS else None,
                'manifest_ok': len(manifests) == 1 if EXPORT_MODELS else None,
                'bundle': str(bundles[0]) if bundles else None,
            }
        )
display(pd.DataFrame(artifact_rows))